# Notebook 1: Healthcare Cost Prediction with Tweedie Loss

## Why the Tweedie Distribution for Healthcare Costs?

Healthcare expenditure data presents a classic **semi-continuous** distribution:
- A large proportion of patients incur **zero costs** in a given period (healthy individuals, no claims)
- Among those who do incur costs, expenditures are **strictly positive** and **heavily right-skewed** (routine visits vs. catastrophic hospitalizations)

The **Tweedie distribution** (power parameter $1 < p < 2$) is a **compound Poisson-Gamma** process that naturally generates exactly this pattern:

$$Y = \sum_{i=1}^{N} X_i, \quad N \sim \text{Poisson}(\lambda), \quad X_i \sim \text{Gamma}(\alpha, \gamma)$$

When $N = 0$, we get $Y = 0$. Otherwise, $Y$ is a sum of Gamma random variables.

The **Tweedie deviance** (loss) for a single observation is:

$$d(y, \mu) = 2 \left[ \frac{y^{2-p}}{(1-p)(2-p)} - \frac{y \cdot \mu^{1-p}}{1-p} + \frac{\mu^{2-p}}{2-p} \right]$$

This notebook demonstrates:
- **Part A**: Standard in-memory models (scikit-learn `TweedieRegressor`, XGBoost)
- **Part B**: Scalable implementations (PyTorch, PySpark, Dask) for large healthcare datasets

---

## Part A: Standard (In-Memory) Healthcare Cost Prediction

### A.1 Environment Setup

In [ ]:
# Install required packages (uncomment if needed)
# !pip install xgboost scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import TweedieRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

np.random.seed(42)
print("Environment ready!")

### A.2 Load and Explore the Medical Cost Dataset

We use the **Medical Cost Personal Dataset** (`insurance.csv`), which contains individual medical charges billed by health insurance. While this dataset has no exact zeros, we will engineer a realistic zero-inflated version to demonstrate the Tweedie distribution for healthcare cost modeling.

In [ ]:
# Load the Medical Cost Personal Dataset
# Download from: https://www.kaggle.com/datasets/mirichoi0218/insurance
# Or use the bundled URL:
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"

try:
    df = pd.read_csv(url)
    print(f"Loaded insurance.csv: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception:
    print("Generating synthetic insurance data...")
    n = 1338
    df = pd.DataFrame({
        'age': np.random.randint(18, 65, n),
        'sex': np.random.choice(['male', 'female'], n),
        'bmi': np.round(np.random.normal(30.6, 6.1, n), 1),
        'children': np.random.choice([0,1,2,3,4,5], n, p=[0.35,0.25,0.2,0.1,0.05,0.05]),
        'smoker': np.random.choice(['yes', 'no'], n, p=[0.2, 0.8]),
        'region': np.random.choice(['southwest','southeast','northwest','northeast'], n),
        'charges': np.zeros(n)
    })
    # Generate realistic charges
    base = 2000 + df['age'] * 200 + (df['smoker']=='yes').astype(int) * 20000
    df['charges'] = np.maximum(base + np.random.exponential(3000, n), 1000)

print(df.head(10))
print(f"\nDataset shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")

### A.3 Exploratory Data Analysis

In [ ]:
# Summary statistics
print("=== Numerical Summary ===")
print(df.describe().round(2))
print(f"\n=== Charges Distribution ===")
print(f"Mean:   ${df['charges'].mean():,.2f}")
print(f"Median: ${df['charges'].median():,.2f}")
print(f"Std:    ${df['charges'].std():,.2f}")
print(f"Min:    ${df['charges'].min():,.2f}")
print(f"Max:    ${df['charges'].max():,.2f}")
print(f"Skew:   {df['charges'].skew():.2f}")
print(f"Kurtosis: {df['charges'].kurtosis():.2f}")

In [ ]:
# Visualize the distribution of charges
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram of charges
axes[0].hist(df['charges'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Medical Charges ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Medical Charges')
axes[0].axvline(df['charges'].mean(), color='red', linestyle='--', label=f"Mean: ${df['charges'].mean():,.0f}")
axes[0].axvline(df['charges'].median(), color='orange', linestyle='--', label=f"Median: ${df['charges'].median():,.0f}")
axes[0].legend()

# Log-transformed histogram
axes[1].hist(np.log1p(df['charges']), bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Log(1 + Charges)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log-Transformed Charges')

# Box plot by smoker status
df.boxplot(column='charges', by='smoker', ax=axes[2])
axes[2].set_title('Charges by Smoking Status')
axes[2].set_xlabel('Smoker')
axes[2].set_ylabel('Charges ($)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('healthcare_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print("Charges are heavily right-skewed - ideal for Tweedie loss!")

### A.4 Creating a Zero-Inflated Healthcare Cost Dataset

Real-world healthcare data often has 30-70% zeros (patients with no claims). We simulate this to demonstrate the Tweedie distribution's key advantage.

In [ ]:
# Create a zero-inflated version of the dataset
# Probability of zero cost depends on age, smoker status, and BMI
df_zi = df.copy()

# Generate zero probability: younger, non-smokers, healthy BMI -> more likely zero
logit = -2.0 + 0.02 * (65 - df_zi['age']) + 1.5 * (df_zi['smoker'] == 'no').astype(float) + 0.03 * (30 - df_zi['bmi']).clip(0)
prob_zero = 1 / (1 + np.exp(-logit))
is_zero = np.random.binomial(1, prob_zero)
df_zi['total_cost'] = df_zi['charges'] * (1 - is_zero)

print(f"=== Zero-Inflated Healthcare Cost Dataset ===")
print(f"Total records:     {len(df_zi)}")
print(f"Zero-cost records: {(df_zi['total_cost'] == 0).sum()} ({(df_zi['total_cost'] == 0).mean()*100:.1f}%)")
print(f"Non-zero mean:     ${df_zi.loc[df_zi['total_cost'] > 0, 'total_cost'].mean():,.2f}")
print(f"Overall mean:      ${df_zi['total_cost'].mean():,.2f}")

# Visualize zero-inflated distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_zi['total_cost'], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Total Healthcare Cost ($)')
ax.set_ylabel('Frequency')
ax.set_title(f'Zero-Inflated Healthcare Costs ({(df_zi["total_cost"]==0).mean()*100:.0f}% zeros)')
ax.annotate(f'{(df_zi["total_cost"]==0).sum()} zeros', xy=(0, (df_zi["total_cost"]==0).sum()),
            fontsize=12, color='red', fontweight='bold')
plt.tight_layout()
plt.savefig('healthcare_zero_inflated.png', dpi=150, bbox_inches='tight')
plt.show()

### A.5 Preprocessing and Train/Test Split

In [ ]:
# Encode categorical variables
le_sex = LabelEncoder()
le_smoker = LabelEncoder()
le_region = LabelEncoder()

df_zi['sex_enc'] = le_sex.fit_transform(df_zi['sex'])
df_zi['smoker_enc'] = le_smoker.fit_transform(df_zi['smoker'])
df_zi['region_enc'] = le_region.fit_transform(df_zi['region'])

# Feature engineering
df_zi['age_squared'] = df_zi['age'] ** 2
df_zi['bmi_smoker'] = df_zi['bmi'] * df_zi['smoker_enc']  # interaction term
df_zi['age_bmi'] = df_zi['age'] * df_zi['bmi']

# Define features and target
feature_cols = ['age', 'sex_enc', 'bmi', 'children', 'smoker_enc', 'region_enc',
                'age_squared', 'bmi_smoker', 'age_bmi']
X = df_zi[feature_cols].values
y = df_zi['total_cost'].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Features:     {feature_cols}")
print(f"\nTarget distribution (train):")
print(f"  Zeros: {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.1f}%)")
print(f"  Mean:  ${y_train.mean():,.2f}")
print(f"  Std:   ${y_train.std():,.2f}")

### A.6 Evaluation Metrics

In [ ]:
def evaluate_model(y_true, y_pred, model_name="Model"):
    """Evaluate regression model with multiple metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # MAPE (avoid division by zero)
    nonzero_mask = y_true > 0
    if nonzero_mask.sum() > 0:
        mape = np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])) * 100
    else:
        mape = float('inf')

    # Tweedie deviance (p=1.5)
    from sklearn.metrics import mean_tweedie_deviance
    y_pred_clipped = np.clip(y_pred, 1e-6, None)  # Tweedie needs positive predictions
    try:
        tweedie_dev = mean_tweedie_deviance(y_true, y_pred_clipped, power=1.5)
    except Exception:
        tweedie_dev = float('nan')

    # Zero prediction accuracy
    pred_zero = (y_pred < 100).sum()  # near-zero threshold
    actual_zero = (y_true == 0).sum()

    print(f"\n{'='*50}")
    print(f"  {model_name} - Evaluation Results")
    print(f"{'='*50}")
    print(f"  MAE:              ${mae:>12,.2f}")
    print(f"  RMSE:             ${rmse:>12,.2f}")
    print(f"  MAPE (non-zero):  {mape:>12.1f}%")
    print(f"  Tweedie Deviance: {tweedie_dev:>12.4f}")
    print(f"  Actual zeros:     {actual_zero:>12d}")
    print(f"  Predicted ~zeros: {pred_zero:>12d}")
    print(f"{'='*50}")

    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'Tweedie_Dev': tweedie_dev}

results = {}

### A.7 Model 1: Scikit-learn TweedieRegressor (GLM)

The `TweedieRegressor` fits a Generalized Linear Model with Tweedie deviance as the loss function. The `log` link ensures positive predictions.

In [ ]:
# Fit TweedieRegressor with different power parameters
powers = [1.2, 1.5, 1.7, 1.9]
best_dev = float('inf')
best_p = None

print("=== Tuning Tweedie Power Parameter ===\n")
for p in powers:
    model = TweedieRegressor(power=p, alpha=0.1, link='log', max_iter=1000)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_pred_clipped = np.clip(y_pred, 1e-6, None)

    from sklearn.metrics import mean_tweedie_deviance
    dev = mean_tweedie_deviance(y_test, y_pred_clipped, power=p)
    mae = mean_absolute_error(y_test, y_pred)
    print(f"  p={p:.1f}: Tweedie Deviance={dev:.4f}, MAE=${mae:,.2f}")

    if dev < best_dev:
        best_dev = dev
        best_p = p

print(f"\n  >>> Best power parameter: p={best_p}")

# Final model with best power
tweedie_glm = TweedieRegressor(power=best_p, alpha=0.1, link='log', max_iter=1000)
tweedie_glm.fit(X_train_scaled, y_train)
y_pred_glm = tweedie_glm.predict(X_test_scaled)

results['Tweedie GLM'] = evaluate_model(y_test, y_pred_glm, f"Tweedie GLM (p={best_p})")

### A.8 Model 2: XGBoost with Tweedie Objective

XGBoost natively supports the `reg:tweedie` objective with the `tweedie_variance_power` hyperparameter.

In [ ]:
# XGBoost with Tweedie objective
xgb_model = xgb.XGBRegressor(
    objective='reg:tweedie',
    tweedie_variance_power=1.5,
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0
)

# Fit with early stopping
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_test)
y_pred_xgb = np.clip(y_pred_xgb, 0, None)  # ensure non-negative

results['XGBoost Tweedie'] = evaluate_model(y_test, y_pred_xgb, "XGBoost Tweedie (p=1.5)")

### A.9 Model Comparison and Visualization

In [ ]:
# Compare predictions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Actual vs Predicted - GLM
axes[0].scatter(y_test, y_pred_glm, alpha=0.4, s=15, color='steelblue')
axes[0].plot([0, y_test.max()], [0, y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Cost ($)')
axes[0].set_ylabel('Predicted Cost ($)')
axes[0].set_title('Tweedie GLM: Actual vs Predicted')
axes[0].set_xlim(-1000, y_test.max() * 1.1)

# Actual vs Predicted - XGBoost
axes[1].scatter(y_test, y_pred_xgb, alpha=0.4, s=15, color='coral')
axes[1].plot([0, y_test.max()], [0, y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Cost ($)')
axes[1].set_ylabel('Predicted Cost ($)')
axes[1].set_title('XGBoost Tweedie: Actual vs Predicted')
axes[1].set_xlim(-1000, y_test.max() * 1.1)

# Residuals comparison
residuals_glm = y_test - y_pred_glm
residuals_xgb = y_test - y_pred_xgb
axes[2].hist(residuals_glm, bins=50, alpha=0.5, color='steelblue', label='Tweedie GLM', density=True)
axes[2].hist(residuals_xgb, bins=50, alpha=0.5, color='coral', label='XGBoost Tweedie', density=True)
axes[2].set_xlabel('Residual ($)')
axes[2].set_ylabel('Density')
axes[2].set_title('Residual Distributions')
axes[2].legend()

plt.tight_layout()
plt.savefig('healthcare_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print("\n=== Model Comparison Summary ===")
summary = pd.DataFrame(results).T
print(summary.to_string())

In [ ]:
# XGBoost feature importance
importance = xgb_model.feature_importances_
feat_imp = pd.Series(importance, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.plot(kind='barh', color='steelblue', ax=ax)
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('XGBoost Tweedie - Feature Importance')
plt.tight_layout()
plt.savefig('healthcare_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Part B: Scaling Healthcare Cost Prediction to Large Data

When healthcare datasets grow to **100M+ rows** (e.g., national claims databases, CMS data), in-memory methods fail. Below we demonstrate three scalable approaches:

1. **PyTorch** - Mini-batch training with custom Tweedie loss
2. **PySpark** - Distributed GLM with Tweedie family
3. **Dask** - Out-of-core fitting with parallel prediction

We first generate a large synthetic dataset mimicking real healthcare cost patterns.

### B.1 Generate Large Synthetic Healthcare Data

In [ ]:
import time
import tracemalloc

def generate_healthcare_data(n_rows, seed=42):
    """Generate synthetic healthcare cost data with zero inflation."""
    rng = np.random.RandomState(seed)

    # Demographics
    age = rng.randint(18, 85, n_rows)
    sex = rng.binomial(1, 0.5, n_rows)
    bmi = np.clip(rng.normal(28, 6, n_rows), 15, 55).round(1)
    smoker = rng.binomial(1, 0.15, n_rows)
    n_conditions = rng.poisson(1.5, n_rows)  # chronic conditions count
    region = rng.randint(0, 4, n_rows)

    # Feature matrix
    X = np.column_stack([age, sex, bmi, smoker, n_conditions, region,
                         age**2, bmi * smoker, age * bmi])

    # Generate cost using compound Poisson-Gamma (Tweedie)
    # mu = exp(linear predictor)
    log_mu = (1.5 + 0.03 * age + 0.5 * smoker + 0.02 * bmi
              + 0.3 * n_conditions + 0.001 * age * bmi * smoker)
    mu = np.exp(np.clip(log_mu, 0, 12))

    # Tweedie parameters (p=1.5)
    p = 1.5
    phi = 2.0
    lam = mu**(2-p) / (phi * (2-p))       # Poisson rate
    alpha_g = (2-p) / (p-1)                # Gamma shape
    beta_g = phi * (p-1) * mu**(p-1)       # Gamma scale

    # Compound Poisson-Gamma generation
    N = rng.poisson(lam)  # number of claims
    y = np.zeros(n_rows)
    for i in range(n_rows):
        if N[i] > 0:
            y[i] = rng.gamma(alpha_g, beta_g[i], N[i]).sum()

    return X.astype(np.float32), y.astype(np.float32)

# Generate datasets of increasing size
for size_label, n in [("Small (10K)", 10_000), ("Medium (100K)", 100_000), ("Large (1M)", 1_000_000)]:
    t0 = time.time()
    X_synth, y_synth = generate_healthcare_data(n)
    elapsed = time.time() - t0
    pct_zero = (y_synth == 0).mean() * 100
    print(f"{size_label:20s}: {n:>10,} rows | {pct_zero:.1f}% zeros | "
          f"Mean=${y_synth[y_synth>0].mean():>10,.2f} | Generated in {elapsed:.2f}s")

print("\nSynthetic data generation ready!")

### B.2 PyTorch: Custom Tweedie Loss with Mini-Batch Training

In [ ]:
# !pip install torch  # uncomment if needed

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


class TweedieLoss(nn.Module):
    """Tweedie deviance loss for PyTorch.

    Implements the negative log-likelihood of the Tweedie distribution:
      loss = -y * mu^(1-p) / (1-p) + mu^(2-p) / (2-p)
    where mu = exp(output) to ensure positive predictions.
    """
    def __init__(self, p=1.5):
        super().__init__()
        assert 1.0 < p < 2.0, "Power parameter must be in (1, 2) for compound Poisson-Gamma"
        self.p = p

    def forward(self, log_mu, y_true):
        mu = torch.exp(log_mu)
        loss = -y_true * torch.pow(mu, 1 - self.p) / (1 - self.p) + \
               torch.pow(mu, 2 - self.p) / (2 - self.p)
        return torch.mean(loss)


class HealthcareCostNet(nn.Module):
    """Feedforward neural network for healthcare cost prediction."""
    def __init__(self, input_dim, hidden_dims=[256, 128, 64]):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(0.2)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)  # returns log(mu)


# Prepare data
N_TRAIN = 500_000
N_TEST = 100_000
X_large, y_large = generate_healthcare_data(N_TRAIN + N_TEST, seed=123)

X_tr = torch.tensor(X_large[:N_TRAIN], dtype=torch.float32)
y_tr = torch.tensor(y_large[:N_TRAIN], dtype=torch.float32)
X_te = torch.tensor(X_large[N_TRAIN:], dtype=torch.float32)
y_te = torch.tensor(y_large[N_TRAIN:], dtype=torch.float32)

# Normalize
mean = X_tr.mean(dim=0)
std = X_tr.std(dim=0) + 1e-8
X_tr = (X_tr - mean) / std
X_te = (X_te - mean) / std

# DataLoader for mini-batch training
train_ds = TensorDataset(X_tr, y_tr)
train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True, num_workers=0)

print(f"PyTorch training set: {N_TRAIN:,} rows")
print(f"PyTorch test set:     {N_TEST:,} rows")
print(f"Input features:       {X_tr.shape[1]}")

In [ ]:
# Train the PyTorch model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

model_pt = HealthcareCostNet(input_dim=X_tr.shape[1]).to(device)
criterion = TweedieLoss(p=1.5)
optimizer = torch.optim.Adam(model_pt.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# Training loop with timing
tracemalloc.start()
t0 = time.time()

train_losses = []
for epoch in range(15):
    model_pt.train()
    epoch_loss = 0.0
    n_batches = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        log_mu = model_pt(X_batch)
        loss = criterion(log_mu, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_pt.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    scheduler.step()
    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)
    if (epoch + 1) % 3 == 0:
        print(f"  Epoch {epoch+1:2d}/15 | Loss: {avg_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.6f}")

training_time_pt = time.time() - t0
_, peak_memory_pt = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"\nPyTorch Training Complete!")
print(f"  Time:        {training_time_pt:.2f}s")
print(f"  Peak Memory: {peak_memory_pt / 1e6:.1f} MB")

# Evaluate
model_pt.eval()
with torch.no_grad():
    log_mu_test = model_pt(X_te.to(device))
    y_pred_pt = torch.exp(log_mu_test).cpu().numpy()

y_pred_pt = np.clip(y_pred_pt, 0, None)
evaluate_model(y_te.numpy(), y_pred_pt, "PyTorch Tweedie NN")

In [ ]:
# Plot training convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, 'b-o', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Tweedie Loss')
axes[0].set_title('PyTorch Training Convergence')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_te.numpy(), y_pred_pt, alpha=0.1, s=5, color='steelblue')
axes[1].plot([0, y_te.max().item()], [0, y_te.max().item()], 'r--', lw=2)
axes[1].set_xlabel('Actual Cost ($)')
axes[1].set_ylabel('Predicted Cost ($)')
axes[1].set_title(f'PyTorch NN: Actual vs Predicted ({N_TEST:,} samples)')
axes[1].set_xlim(0, np.percentile(y_te.numpy(), 99))
axes[1].set_ylim(0, np.percentile(y_pred_pt, 99))

plt.tight_layout()
plt.savefig('healthcare_pytorch_results.png', dpi=150, bbox_inches='tight')
plt.show()

### B.3 PySpark: Distributed Tweedie GLM

PySpark's `GeneralizedLinearRegression` natively supports the Tweedie family, enabling distributed training on clusters.

In [ ]:
# !pip install pyspark  # uncomment if needed

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GeneralizedLinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Create local Spark session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("HealthcareTweedie") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print(f"Cores: {spark.sparkContext.defaultParallelism}")

In [ ]:
# Generate large synthetic data and load into Spark
N_SPARK = 1_000_000
X_spark, y_spark = generate_healthcare_data(N_SPARK, seed=456)

feature_names = ['age', 'sex', 'bmi', 'smoker', 'n_conditions', 'region',
                 'age_sq', 'bmi_smoker', 'age_bmi']

# Create pandas DataFrame, then convert to Spark
pdf = pd.DataFrame(X_spark, columns=feature_names)
pdf['total_cost'] = y_spark

tracemalloc.start()
t0 = time.time()

sdf = spark.createDataFrame(pdf)
sdf = sdf.repartition(8)

# Assemble features into a single vector column
assembler = VectorAssembler(inputCols=feature_names, outputCol="features")
sdf = assembler.transform(sdf)

# Train/test split
train_sdf, test_sdf = sdf.randomSplit([0.8, 0.2], seed=42)
train_sdf.cache()
test_sdf.cache()

print(f"Spark training rows: {train_sdf.count():,}")
print(f"Spark test rows:     {test_sdf.count():,}")

In [ ]:
# Fit Tweedie GLM in PySpark
# family="tweedie", variancePower=1.5, linkPower=0 (log link)
glr = GeneralizedLinearRegression(
    family="tweedie",
    variancePower=1.5,
    linkPower=0,           # 0 = log link
    featuresCol="features",
    labelCol="total_cost",
    maxIter=100,
    regParam=0.01
)

spark_model = glr.fit(train_sdf)
training_time_spark = time.time() - t0
_, peak_memory_spark = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"PySpark Tweedie GLM Training Complete!")
print(f"  Time:        {training_time_spark:.2f}s")
print(f"  Peak Memory: {peak_memory_spark / 1e6:.1f} MB")

# Model summary
summary = spark_model.summary
print(f"  AIC:         {summary.aic:.2f}")
print(f"  Deviance:    {summary.deviance:.2f}")
print(f"  Null Dev:    {summary.nullDeviance:.2f}")

# Coefficients
print(f"\nCoefficients:")
for name, coef in zip(feature_names, spark_model.coefficients):
    print(f"  {name:15s}: {coef:>10.4f}")
print(f"  {'Intercept':15s}: {spark_model.intercept:>10.4f}")

In [ ]:
# Evaluate PySpark model
predictions = spark_model.transform(test_sdf)

# Collect predictions for metrics
pred_pdf = predictions.select("total_cost", "prediction").toPandas()
y_true_spark = pred_pdf['total_cost'].values
y_pred_spark = np.clip(pred_pdf['prediction'].values, 0, None)

evaluate_model(y_true_spark, y_pred_spark, "PySpark Tweedie GLM")

# Clean up
spark.stop()
print("\nSpark session stopped.")

### B.4 Dask: Out-of-Core Tweedie Regression

Dask enables out-of-core fitting by partitioning the data and wrapping scikit-learn models for parallel prediction.

In [ ]:
# !pip install dask dask-ml  # uncomment if needed

import dask.dataframe as dd
import dask.array as da
from dask.distributed import Client, LocalCluster
from dask_ml.wrappers import ParallelPostFit

# Start local Dask cluster
cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit='1GB')
client = Client(cluster)
print(f"Dask Dashboard: {client.dashboard_link}")
print(f"Workers: {len(client.scheduler_info()['workers'])}")

In [ ]:
# Generate partitioned data with Dask
N_DASK = 2_000_000

tracemalloc.start()
t0 = time.time()

# Generate in chunks to simulate out-of-core
chunk_size = 500_000
dfs = []
for i in range(0, N_DASK, chunk_size):
    X_chunk, y_chunk = generate_healthcare_data(chunk_size, seed=i)
    feat_names = ['age', 'sex', 'bmi', 'smoker', 'n_conditions', 'region',
                  'age_sq', 'bmi_smoker', 'age_bmi']
    chunk_df = pd.DataFrame(X_chunk, columns=feat_names)
    chunk_df['total_cost'] = y_chunk
    dfs.append(chunk_df)

dask_df = dd.from_pandas(pd.concat(dfs, ignore_index=True), npartitions=8)
print(f"Dask DataFrame: {len(dask_df):,} rows in {dask_df.npartitions} partitions")
print(f"Partition size:  ~{len(dask_df) // dask_df.npartitions:,} rows each")

# Split features and target
X_dask = dask_df[feat_names].values.compute()
y_dask = dask_df['total_cost'].values.compute()

# Train/test split
split_idx = int(0.8 * len(X_dask))
X_train_dask, X_test_dask = X_dask[:split_idx], X_dask[split_idx:]
y_train_dask, y_test_dask = y_dask[:split_idx], y_dask[split_idx:]

print(f"Training: {len(X_train_dask):,} | Test: {len(X_test_dask):,}")

In [ ]:
# Fit TweedieRegressor and wrap with ParallelPostFit for distributed prediction
from sklearn.linear_model import TweedieRegressor as SkTweedie
from sklearn.preprocessing import StandardScaler as SkScaler

# Scale features
scaler_dask = SkScaler()
X_train_dask_scaled = scaler_dask.fit_transform(X_train_dask)
X_test_dask_scaled = scaler_dask.transform(X_test_dask)

# Fit on a subsample (Dask handles prediction at scale)
subsample_idx = np.random.choice(len(X_train_dask_scaled), size=min(500_000, len(X_train_dask_scaled)), replace=False)
tweedie_dask = SkTweedie(power=1.5, alpha=0.1, link='log', max_iter=500)
tweedie_dask.fit(X_train_dask_scaled[subsample_idx], y_train_dask[subsample_idx])

# Wrap for parallel prediction
parallel_model = ParallelPostFit(tweedie_dask)

# Predict using Dask arrays (distributed)
X_test_da = da.from_array(X_test_dask_scaled, chunks=(100_000, -1))
y_pred_dask = parallel_model.predict(X_test_da).compute()

training_time_dask = time.time() - t0
_, peak_memory_dask = tracemalloc.get_traced_memory()
tracemalloc.stop()

y_pred_dask = np.clip(y_pred_dask, 0, None)
evaluate_model(y_test_dask, y_pred_dask, "Dask Tweedie GLM")

print(f"\nDask Pipeline Complete!")
print(f"  Time:        {training_time_dask:.2f}s")
print(f"  Peak Memory: {peak_memory_dask / 1e6:.1f} MB")

# Shutdown
client.close()
cluster.close()

### B.5 Scalable Framework Comparison

In [ ]:
# Summary comparison
benchmark = pd.DataFrame({
    'Framework': ['PyTorch NN', 'PySpark GLM', 'Dask GLM'],
    'Dataset Size': [f'{N_TRAIN:,}', f'{N_SPARK:,}', f'{N_DASK:,}'],
    'Training Time (s)': [f'{training_time_pt:.1f}', f'{training_time_spark:.1f}', f'{training_time_dask:.1f}'],
    'Peak Memory (MB)': [f'{peak_memory_pt/1e6:.0f}', f'{peak_memory_spark/1e6:.0f}', f'{peak_memory_dask/1e6:.0f}'],
    'Best For': [
        'Complex non-linear patterns, GPU acceleration',
        'Massive distributed clusters, SQL integration',
        'Out-of-core on single machine, scikit-learn compatible'
    ]
})

print("\n" + "="*80)
print("  SCALABLE FRAMEWORK COMPARISON - Healthcare Cost Prediction")
print("="*80)
print(benchmark.to_string(index=False))
print("="*80)
print("\nKey Takeaways:")
print("  - PyTorch: Best for complex patterns; GPU scales to billions of rows")
print("  - PySpark: Best for cluster-scale data; native Tweedie GLM support")
print("  - Dask:    Best for out-of-core on a single machine; minimal code changes")

---
## Summary

This notebook demonstrated the Tweedie distribution for healthcare cost prediction at two scales:

- **Part A** showed that Tweedie GLMs and XGBoost with Tweedie objective handle zero-inflated medical charges effectively, outperforming standard regression approaches.
- **Part B** showed that the same Tweedie loss function scales seamlessly to millions of rows using PyTorch (mini-batch + GPU), PySpark (distributed GLM), and Dask (out-of-core parallel prediction).

The Tweedie distribution's compound Poisson-Gamma structure makes it the natural choice for healthcare cost data, where many patients incur zero costs and those who do face right-skewed expenditures.